# ⚖️ Step 6: Class Imbalance Strategy Comparison on Kaggle GPU

### 📌 Overview & Setup Instructions
- **Project**: Federated Medical AI System (Step 6 of 20)
- **Objective**: Rigorously compare 3 techniques to solve medical class imbalance on 6,000 RSNA DICOM images:
  1. **Weighted BCE Loss** (`pos_weight = neg / pos`)
  2. **Focal Loss** ($\\gamma=2.0, \\alpha=0.778$)
  3. **Weighted Random Over-Sampling**
- **Hardware Requirement**: **Kaggle GPU T4 x2** (In Kaggle right sidebar: Settings -> Accelerator -> Select **GPU T4 x2**. Do NOT select GPU P100).
- **Dataset Dependency**: Attach Kaggle Dataset `rsna-pneumonia-detection-challenge` (`/kaggle/input/rsna-pneumonia-detection-challenge`).
- **Expected GPU Wall-Clock Runtime**: **~15 to 22 minutes** (10 full epochs per strategy).

---

> [!IMPORTANT]
> **GPU Selection Note**: Please select **GPU T4 x2** in Kaggle settings. Tesla P100 GPUs use compute capability `sm_60` which is unsupported in modern PyTorch builds.
> **No Synthetic Data Fallback**: This notebook strictly loads real RSNA DICOM images. If the dataset is not attached, the notebook will stop with an error.



In [ ]:
# Cell 1: Environment Setup & Hardware Disclosure
!pip install -q pydicom torchvision scikit-learn matplotlib pandas numpy opencv-python

import os
import sys
import time
import json
import copy
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import pydicom

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix

print("=== SYSTEM & HARDWARE DISCLOSURE ===")
print(f"PyTorch Version   : {torch.__version__}")
print(f"CUDA Available?   : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU Device Name   : {gpu_name}")
    
    try:
        test_tensor = torch.zeros(1).cuda()
        print(f"GPU Tensor Check  : SUCCESS (Compute capability supported on {gpu_name})")
        device = torch.device("cuda")
    except Exception as e:
        print()
        print("!" * 80)
        print("CRITICAL GPU COMPATIBILITY ERROR DETECTED:")
        print(f"  {e}")
        print("REASON: Kaggle Tesla P100 (compute capability sm_60) is incompatible with modern PyTorch builds.")
        print("ACTION REQUIRED: In Kaggle's right-hand panel, under Settings -> Accelerator:")
        print("                 Switch accelerator from 'GPU P100' to 'GPU T4 x2'.")
        print("!" * 80)
        print()
        raise RuntimeError("Incompatible GPU (Tesla P100). Please switch Kaggle Accelerator setting to 'GPU T4 x2'.")
else:
    print("WARNING: CUDA Not Available! Please attach GPU accelerator in Kaggle Settings.")
    device = torch.device("cpu")

OUTPUT_DIR = Path("/kaggle/working/outputs")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
HEATMAPS_DIR = OUTPUT_DIR / "heatmaps"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
HEATMAPS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output Directory  : {OUTPUT_DIR}")

def get_paths(dataset_name="federated-medical-ai-outputs"):
    """
    Centralized path resolution engine. Checks /kaggle/input/[dataset_name]/ FIRST for pre-saved
    checkpoints, partition files, or outputs before assuming a step needs retraining.
    """
    input_base = Path(f"/kaggle/input/{dataset_name}")
    if input_base.exists():
        print(f"[OK] Discovered attached dataset at: {input_base}")
        return {
            "output_dir": OUTPUT_DIR,
            "checkpoint_dir": input_base / "checkpoints" if (input_base / "checkpoints").exists() else CHECKPOINT_DIR,
            "partition_dir": input_base / "client_partitions" if (input_base / "client_partitions").exists() else OUTPUT_DIR / "client_partitions",
            "is_attached": True
        }

    input_dirs = list(Path("/kaggle/input").glob("**/checkpoints"))
    if len(input_dirs) > 0:
        matched_dir = input_dirs[0]
        matched_parent = matched_dir.parent
        ds_name = matched_parent.parts[3] if len(matched_parent.parts) > 3 else "attached-dataset"
        print(f"[OK] Discovered attached dataset containing checkpoints at: /kaggle/input/{ds_name}/checkpoints/")
        return {
            "output_dir": OUTPUT_DIR,
            "checkpoint_dir": matched_dir,
            "partition_dir": matched_parent / "client_partitions" if (matched_parent / "client_partitions").exists() else OUTPUT_DIR / "client_partitions",
            "is_attached": True
        }

    return {
        "output_dir": OUTPUT_DIR,
        "checkpoint_dir": CHECKPOINT_DIR,
        "partition_dir": OUTPUT_DIR / "client_partitions",
        "is_attached": False
    }

def find_checkpoint(filename, dataset_name="federated-medical-ai-outputs"):
    """
    Checks /kaggle/input/ attached datasets FIRST before assuming a checkpoint needs retraining.
    Returns the resolved Path object.
    """
    working_path = CHECKPOINT_DIR / filename
    if working_path.exists():
        print(f"[CACHE HIT] Found checkpoint in local session working dir: {working_path}")
        return working_path

    input_matches = list(Path("/kaggle/input").glob(f"**/{filename}"))
    if len(input_matches) > 0:
        found_path = input_matches[0]
        ds_name = found_path.parts[3] if len(found_path.parts) > 3 else dataset_name
        print(f"[OK] [CACHE HIT] Found pre-saved checkpoint in attached Kaggle dataset!")
        print(f"     Loaded from: /kaggle/input/{ds_name}/checkpoints/{filename}")
        print(f"     To reuse in future sessions: Add Input -> search for {ds_name} -> Add, then load from /kaggle/input/{ds_name}/checkpoints/{filename}")
        print("     >> Reusing prior verified model weights without retraining! <<")
        return found_path

    print(f"[INFO] Checkpoint '{filename}' not found in /kaggle/input/ attached datasets or local working dir.")
    return working_path

def find_client_partitions():
    """
    Checks /kaggle/input/ attached datasets FIRST for 5-client Dirichlet partitions before regenerating.
    """
    working_dir = OUTPUT_DIR / "client_partitions"
    if working_dir.exists() and len(list(working_dir.glob("client_*.csv"))) == 5:
        print(f"[CACHE HIT] Found 5 client partition files in local working dir: {working_dir}")
        return working_dir

    input_matches = list(Path("/kaggle/input").glob("**/client_partitions"))
    for match in input_matches:
        if len(list(match.glob("client_*.csv"))) == 5:
            ds_name = match.parts[3] if len(match.parts) > 3 else "attached-dataset"
            print(f"[OK] [CACHE HIT] Found pre-saved client partitions in attached dataset!")
            print(f"     Loaded from: /kaggle/input/{ds_name}/client_partitions/")
            return match

    return working_dir



In [ ]:
# Cell 2: RSNA Data Mount Verification (Strict Error Check)
RSNA_DATA_DIR = Path("/kaggle/input/rsna-pneumonia-detection-challenge")
LABELS_CSV = RSNA_DATA_DIR / "stage_2_train_labels.csv"

if not LABELS_CSV.exists():
    alt_paths = list(Path("/kaggle/input").glob("**/stage_2_train_labels.csv"))
    if len(alt_paths) > 0:
        LABELS_CSV = alt_paths[0]
        RSNA_DATA_DIR = LABELS_CSV.parent
        print(f"[OK] Found RSNA Labels CSV at: {LABELS_CSV}")
    else:
        raise FileNotFoundError(
            f"CRITICAL ERROR: RSNA Dataset not found at {RSNA_DATA_DIR}! "
            "Please add dataset 'rsna-pneumonia-detection-challenge' to this Kaggle notebook before running."
        )

IMAGES_DIR = RSNA_DATA_DIR / "stage_2_train_images"
if not IMAGES_DIR.exists():
    alt_imgs = list(RSNA_DATA_DIR.glob("**/stage_2_train_images"))
    if len(alt_imgs) > 0:
        IMAGES_DIR = alt_imgs[0]

print(f"[OK] RSNA Labels CSV : {LABELS_CSV}")
print(f"[OK] RSNA Images Dir : {IMAGES_DIR}")



In [ ]:
# Cell 3: DICOM PyTorch Dataset & Splitter
def parse_and_split_rsna(labels_csv_path, subset_size=6000, seed=42):
    df_raw = pd.read_csv(labels_csv_path)
    grouped = []
    for pid, group in df_raw.groupby("patientId"):
        target = group["Target"].iloc[0]
        grouped.append({"patientId": pid, "Target": int(target)})
    df_unique = pd.DataFrame(grouped)

    if subset_size and len(df_unique) > subset_size:
        df_unique = df_unique.sample(n=subset_size, random_state=seed).reset_index(drop=True)

    shuffled = df_unique.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    n_total = len(shuffled)
    n_train = int(n_total * 0.70)
    n_val = int(n_total * 0.15)

    train_df = shuffled.iloc[:n_train].reset_index(drop=True)
    val_df = shuffled.iloc[n_train:n_train + n_val].reset_index(drop=True)
    test_df = shuffled.iloc[n_train + n_val:].reset_index(drop=True)

    print(f"Data Split Summary (Subset Size = {len(df_unique)} Patients):")
    print(f"  - Train : {len(train_df)} patients ({train_df['Target'].mean()*100:.2f}% positive)")
    print(f"  - Val   : {len(val_df)} patients ({val_df['Target'].mean()*100:.2f}% positive)")
    print(f"  - Test  : {len(test_df)} patients ({test_df['Target'].mean()*100:.2f}% positive)")
    return train_df, val_df, test_df

class RSNADICOMDataset(Dataset):
    def __init__(self, df, images_dir, image_size=(224, 224)):
        self.df = df.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.image_size = image_size
        self.transform = transforms.Compose([
            transforms.Resize(image_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        pid = row["patientId"]
        target = int(row.get("Target", 0))

        dcm_path = self.images_dir / f"{pid}.dcm"
        if not dcm_path.exists():
            png_path = self.images_dir / f"{pid}.png"
            if png_path.exists():
                img = Image.open(png_path).convert("RGB")
            else:
                raise FileNotFoundError(f"DICOM image not found for patient {pid} at {dcm_path}")
        else:
            dcm = pydicom.dcmread(str(dcm_path))
            arr = dcm.pixel_array.astype(np.float32)
            arr_min, arr_max = arr.min(), arr.max()
            if arr_max > arr_min:
                arr = (arr - arr_min) / (arr_max - arr_min) * 255.0
            else:
                arr = np.zeros_like(arr)
            img = Image.fromarray(arr.astype(np.uint8)).convert("RGB")

        tensor = self.transform(img)
        return tensor, torch.tensor(target, dtype=torch.float32)

train_df, val_df, test_df = parse_and_split_rsna(LABELS_CSV, subset_size=6000, seed=42)

train_dataset = RSNADICOMDataset(train_df, IMAGES_DIR)
val_dataset = RSNADICOMDataset(val_df, IMAGES_DIR)
test_dataset = RSNADICOMDataset(test_df, IMAGES_DIR)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)



In [ ]:
# Cell 4: Model Architecture & Evaluation Helper
def build_resnet18(pretrained=True):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pretrained else None)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, 1)
    )
    return model

def evaluate_model(model, loader, device):
    model.eval()
    criterion = nn.BCEWithLogitsLoss()
    total_loss = 0.0
    all_targets = []
    all_probs = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            logits = model(images).squeeze(-1)
            loss = criterion(logits, labels)

            total_loss += loss.item() * len(labels)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_targets.extend(labels.cpu().numpy().tolist())
            all_probs.extend(probs.tolist())

    avg_loss = total_loss / max(1, len(all_targets))
    auc = float(roc_auc_score(all_targets, all_probs)) if len(np.unique(all_targets)) > 1 else 0.5
    return avg_loss, auc, all_targets, all_probs



In [ ]:
# Cell 4: Custom Loss Functions & Weighted Sampler Builders
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.778, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        p_t = probs * targets + (1 - probs) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_loss = alpha_t * ((1 - p_t) ** self.gamma) * bce
        return focal_loss.mean()

def get_weighted_sampler(df):
    targets = df["Target"].values
    class_counts = np.bincount(targets)
    class_weights = 1.0 / np.maximum(1, class_counts)
    sample_weights = class_weights[targets]
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )
    return sampler



In [ ]:
# Cell 5: Comparative Training Loop Across 3 Imbalance Techniques (10 Epochs Each)
def train_imbalance_experiment(strategy_name, criterion_fn, sampler=None, epochs=10):
    print()
    print("=" * 70)
    print(f"   STARTING EXPERIMENT: {strategy_name}")
    print("=" * 70)
    
    torch.manual_seed(42)
    model = build_resnet18(pretrained=True).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

    if sampler is not None:
        curr_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler, num_workers=2, pin_memory=True)
    else:
        curr_loader = train_loader

    history = []
    best_val_auc = 0.0
    best_model_weights = None

    start_time = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        total_samples = 0

        for images, labels in curr_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            logits = model(images).squeeze(-1)
            loss = criterion_fn(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * len(labels)
            total_samples += len(labels)

        train_loss = running_loss / max(1, total_samples)
        val_loss, val_auc, _, _ = evaluate_model(model, val_loader, device)

        history.append({
            "epoch": epoch,
            "train_loss": round(train_loss, 4),
            "val_loss": round(val_loss, 4),
            "val_auc": round(val_auc, 4)
        })

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_model_weights = copy.deepcopy(model.state_dict())
            tag = " [BEST]"
        else:
            tag = ""

        print(f"[{strategy_name}] Epoch [{epoch:02d}/{epochs:02d}] - Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val AUC: {val_auc:.4f}{tag}")

    elapsed = time.time() - start_time
    
    model.load_state_dict(best_model_weights)
    test_loss, test_auc, test_targets, test_probs = evaluate_model(model, test_loader, device)
    test_preds = (np.array(test_probs) >= 0.5).astype(int)

    f1 = float(f1_score(test_targets, test_preds, zero_division=0))
    precision = float(precision_score(test_targets, test_preds, zero_division=0))
    recall = float(recall_score(test_targets, test_preds, zero_division=0))
    cm = confusion_matrix(test_targets, test_preds)
    spec = float(cm[0,0] / max(1, cm[0,0] + cm[0,1]))

    metrics = {
        "strategy": strategy_name,
        "elapsed_mins": round(elapsed / 60.0, 2),
        "test_auc": round(test_auc, 4),
        "test_f1": round(f1, 4),
        "test_precision": round(precision, 4),
        "test_recall": round(recall, 4),
        "test_specificity": round(spec, 4),
        "history": history
    }
    return metrics, model

pos_count = train_df["Target"].sum()
neg_count = len(train_df) - pos_count
pos_weight_val = neg_count / float(max(1, pos_count))
pos_weight_tensor = torch.tensor([pos_weight_val], dtype=torch.float32).to(device)

criterion_weighted_bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
criterion_focal = FocalLoss(alpha=0.778, gamma=2.0)
criterion_standard = nn.BCEWithLogitsLoss()
sampler_weighted = get_weighted_sampler(train_df)

res_wbce, model_wbce = train_imbalance_experiment("Weighted_BCE", criterion_weighted_bce, epochs=10)
res_focal, model_focal = train_imbalance_experiment("Focal_Loss", criterion_focal, epochs=10)
res_sampler, model_sampler = train_imbalance_experiment("Weighted_Sampler", criterion_standard, sampler=sampler_weighted, epochs=10)



In [ ]:
# Cell 6: Comparative Analysis, Tables & Visualizations
all_results = [res_wbce, res_focal, res_sampler]

print()
print("=" * 80)
print("              STEP 6 CLASS IMBALANCE COMPARATIVE SUMMARY")
print("=" * 80)
summary_df = pd.DataFrame([
    {
        "Strategy": r["strategy"],
        "ROC-AUC": r["test_auc"],
        "F1-Score": r["test_f1"],
        "Recall (Sensitivity)": r["test_recall"],
        "Specificity": r["test_specificity"],
        "Precision": r["test_precision"],
        "Runtime (mins)": r["elapsed_mins"]
    }
    for r in all_results
])
print(summary_df.to_string(index=False))
print("=" * 80)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

colors = {'Weighted_BCE': 'blue', 'Focal_Loss': 'crimson', 'Weighted_Sampler': 'green'}

for r in all_results:
    name = r["strategy"]
    epochs = [h["epoch"] for h in r["history"]]
    val_losses = [h["val_loss"] for h in r["history"]]
    val_aucs = [h["val_auc"] for h in r["history"]]
    
    ax1.plot(epochs, val_losses, '-o', color=colors[name], label=f'{name} Val Loss')
    ax2.plot(epochs, val_aucs, '-s', color=colors[name], label=f'{name} Val AUC (Test={r["test_auc"]})')

ax1.set_title('Val Loss Progression by Imbalance Strategy', fontsize=12, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend()

ax2.set_title('Val ROC-AUC Progression by Imbalance Strategy', fontsize=12, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('ROC-AUC')
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend()

plt.tight_layout()
plot_path = OUTPUT_DIR / "imbalance_strategies_comparison.png"
plt.savefig(plot_path, dpi=200)
plt.show()

with open(OUTPUT_DIR / "results_step6.json", "w") as f:
    json.dump(all_results, f, indent=4)

print()
print(f"[OK] Step 6 Results saved to {OUTPUT_DIR / 'results_step6.json'}")



---

### 📋 Manual Execution Checklist & Logging Table

Record your live Kaggle GPU session results for Step 6:

| Strategy | Verified Test ROC-AUC | Test F1-Score | Test Recall | Test Specificity | Screenshot Taken? |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **1. Weighted BCE** | `0.____` | `0.____` | `0.____` | `0.____` | [ ] Yes |
| **2. Focal Loss ($\\gamma=2$)** | `0.____` | `0.____` | `0.____` | `0.____` | [ ] Yes |
| **3. Weighted Sampler** | `0.____` | `0.____` | `0.____` | `0.____` | [ ] Yes |

- **Best Strategy Selected**: `____________________`
- **Plot Saved**: `outputs/imbalance_strategies_comparison.png`



---

### 💾 Kaggle Output Persistence & Cross-Session Dataset Saving

> [!IMPORTANT]
> Kaggle's `/kaggle/working` directory is **ephemeral** and cleared when a session ends.
> To persist model checkpoints, client partitions, plots, and markdown reports across separate Kaggle sessions without retraining:
> 1. Click **Save Version** (top right menu) $\rightarrow$ Select **Save & Run All (Commit)** $\rightarrow$ Click **Save**.
> 2. Once completed, navigate to your notebook output page $\rightarrow$ Click **Create Dataset** (e.g., name it `federated-medical-ai-outputs`).
> 3. In future sessions (e.g., Step 7 Grad-CAM or Step 11 FedProx): Click **+ Add Input** $\rightarrow$ Search for `federated-medical-ai-outputs` $\rightarrow$ Click **Add**.
> 4. The notebooks will automatically discover `/kaggle/input/federated-medical-ai-outputs/checkpoints/` and load pre-saved weights without retraining!



In [ ]:
# Cell: Kaggle Output Persistence & Dataset Auto-Packager
import zipfile

zip_path = Path("/kaggle/working/outputs_bundle.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file in OUTPUT_DIR.rglob("*"):
        if file.is_file() and file.name != "outputs_bundle.zip":
            arcname = file.relative_to(OUTPUT_DIR)
            zipf.write(file, arcname)

print("=" * 85)
print("  KAGGLE OUTPUT PERSISTENCE & CROSS-SESSION REUSE INSTRUCTIONS")
print("=" * 85)
print(f"[OK] Successfully packaged all checkpoints, plots, and reports into: {zip_path}")
print()
print("To reuse this checkpoint / dataset in future Kaggle sessions:")
print(" 1. In top right notebook menu: Click 'Save Version' -> Select 'Save & Run All' -> Save.")
print(" 2. OR go to your notebook output page -> Click 'Create Dataset' -> Name it 'federated-medical-ai-outputs'.")
print(" 3. In future sessions (e.g., Step 7 Grad-CAM or Step 11 FedProx):")
print("    - Click '+ Add Input' in the right sidebar -> Search for 'federated-medical-ai-outputs' -> Click 'Add'.")
print("    - Notebooks will automatically detect pre-saved checkpoints from:")
print("      /kaggle/input/federated-medical-ai-outputs/checkpoints/...")
print("      and reuse real model weights without silently retraining!")
print("=" * 85)

